# चुनौती: डेटा विज्ञानको बारेमा पाठ विश्लेषण गर्ने

यस उदाहरणमा, हामी एउटा साधारण अभ्यास गर्नेछौं जसले पारम्परिक डेटा विज्ञान प्रक्रियाका सबै चरणहरू समेट्छ। तपाईंले कुनै पनि कोड लेख्न आवश्यक छैन, तपाईं तलको सेलहरूमा क्लिक गरेर तिनीहरूलाई कार्यान्वयन गर्न र परिणाम अवलोकन गर्नसक्नुहुन्छ। एक चुनौतीको रूपमा, तपाईंलाई फरक डेटा प्रयोग गरेर यस कोडलाई प्रयास गर्न प्रोत्साहित गरिएको छ। 

## लक्ष्य

यस पाठमा, हामीले डेटा विज्ञानसँग सम्बन्धित विभिन्न अवधारणाहरू छलफल गरिरहेका छौं। हामी **पाठ खनन** गरेर थप सम्बन्धित अवधारणाहरू पत्ता लगाउने प्रयास गर्नेछौं। हामी डेटा विज्ञानको बारेमा एउटा पाठबाट सुरुवात गर्नेछौं, त्यसबाट मुख्य शब्दहरू निकाल्ने छौं, र त्यसपछि परिणामलाई दृश्यात्मक बनाउन कोसिस गर्नेछौं।

पाठको रूपमा, म विकिपिडियाबाट डेटा विज्ञानको पृष्ठ प्रयोग गर्नेछु:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## चरण 1: डाटा प्राप्त गर्दै

प्रत्येक डाटा साईन्स प्रक्रियामा पहिलो चरण भनेको डाटा प्राप्त गर्नु हो। हामी त्यसका लागि `requests` लाइब्रेरी प्रयोग गर्नेछौं:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## चरण २: डाटालाई परिवर्तन गर्दै

अर्को चरण भनेको डाटालाई प्रशोधनका लागि उपयुक्त स्वरूपमा परिणत गर्नु हो। हाम्रो अवस्थामा, हामीले पृष्ठबाट HTML स्रोत कोड डाउनलोड गरेका छौं, र यसलाई सामान्य पाठमा रूपान्तरण गर्नु आवश्यक छ।

यो गर्नका लागि धेरै तरिका छन्। हामी [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/) प्रयोग गर्नेछौं, जुन HTML पार्सिंगका लागि लोकप्रिय Python पुस्तकालय हो। BeautifulSoup ले हामीलाई निश्चित HTML तत्वहरू लक्षित गर्न अनुमति दिन्छ, त्यसैले हामी विकिपीडियाको मुख्य लेख सामग्रीमा ध्यान केन्द्रित गर्न सक्छौं र केही नेभिगेसन मेनुहरू, साइडबारहरू, फूटरहरू, र अन्य अप्रासंगिक सामग्रीहरू घटाउन सक्छौं (यद्यपि केही बोइलरप्लेट पाठ अझै बाँकी हुन सक्छ)।


पहिलो, हामीलाई HTML पार्सिंगका लागि BeautifulSoup लाइब्रेरी स्थापना गर्न आवश्यक छ:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## चरण ३: अन्तर्दृष्टि प्राप्त गर्दै

सबैभन्दा महत्वपूर्ण चरण भनेको हाम्रा डाटालाई यस्तो रूप दिने हो जहाँबाट हामी अन्तर्दृष्टिहरू निकाल्न सक्छौं। हाम्रो मामलामा, हामी पाठबाट कुञ्जीशब्दहरू निकाल्न चाहन्छौं, र कुन कुञ्जीशब्दहरू बढी अर्थपूर्ण छन् भनेर हेर्न चाहन्छौं।

हामी कुञ्जीशब्द निकाल्नका लागि [RAKE](https://github.com/aneesha/RAKE) नामक Python पुस्तकालय प्रयोग गर्नेछौं। पहिलो, यदि यो पुस्तकालय उपलब्ध छैन भने यसलाई स्थापना गरौं: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

मुख्य कार्यक्षमता `Rake` वस्तुबाट उपलब्ध छ, जसलाई हामी केहि प्यारामिटरहरूको प्रयोग गरेर अनुकूलित गर्न सक्छौं। हाम्रो अवस्थामा, हामी कुञ्जीशब्दको न्यूनतम लम्बाइ ५ अक्षर, दस्तावेजमा कुञ्जीशब्दको न्यूनतम आवृत्ति ३, र कुञ्जीशब्दमा शब्दहरूको अधिकतम संख्या २ सेट गर्नेछौं। अन्य मानहरूसँग खेल्न स्वतन्त्र महसुस गर्नुहोस् र नतिजा अवलोकन गर्नुहोस्।


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


हामीले सम्बन्धित महत्त्वको डिग्रीसँगै सर्तहरूको सूची प्राप्त गर्यौं। तपाईँले देख्न सक्नुहुन्छ, सबैभन्दा सम्बन्धित विषयहरु, जस्तै मेशिन लर्निंग र ठुलो डेटा, सूचीमा शीर्ष स्थानहरूमा छन्।

## चरण ४: नतिजा दृश्यात्मक बनाउने

मानिसहरूले डेटा सबैभन्दा राम्रोसँग दृश्यात्मक रूपमा व्याख्या गर्न सक्छन्। त्यसैले केही अन्तर्दृष्टि प्राप्त गर्न डेटालाई दृश्यात्मक बनाउनु प्रायः अर्थपूर्ण हुन्छ। हामी Python मा `matplotlib` पुस्तकालय प्रयोग गरेर सम्बन्धितता सहित कुञ्जीशब्दहरूको सरल वितरणलाई प्लट गर्न सक्छौं:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

शब्द आवृत्तिहरूलाई अझ राम्रो तरिकाले दृश्यात्मक बनाउन, **Word Cloud** प्रयोग गर्न सकिन्छ। हाम्रो कीवर्ड सूचीबाट शब्द बादल प्लट गर्न हामीले अर्को पुस्तकालय स्थापना गर्नुपर्नेछ।


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` वस्तु मूल पाठ वा शब्दहरूको पहिलेबाट गणना गरिएको सूची जसमा तिनीहरूको आवृत्तिहरू हुन्छन्, लिन जिम्मेवार हुन्छ, र एक छवि फर्काउँछ, जुन पछि `matplotlib` प्रयोग गरेर प्रदर्शन गर्न सकिन्छ:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

हामीले `WordCloud` मा मूल पाठ पनि पास गर्न सक्छौं - हेरौं कि हामी समान नतिजा प्राप्त गर्न सक्षम छौं कि छैनौं:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

तपाईंले देख्न सक्नुहुन्छ कि शब्द बादल अब बढी प्रभावशाली देखिन्छ, तर यसमा धेरै अनावश्यक शब्दहरू पनि छन् (जस्तै `Retrieved on` जस्ता असम्बन्धित शब्दहरू)। साथै, हामीलाई दुई शब्दहरू मिलेर बनेका कम कीवर्डहरू प्राप्त हुन्छन्, जस्तै *data scientist*, वा *computer science*। यो यसैले हो कि RAKE एल्गोरिदमले पाठबाट राम्रो कीवर्ड चयन गर्न धेरै राम्रो काम गर्छ। यो उदाहरणले डाटा पूर्व-प्रक्रिया र सफाईको महत्त्वलाई देखाउँछ, किनभने अन्त्यमा स्पष्ट चित्रले हामीलाई राम्रो निर्णय लिन सक्षम बनाउँछ।

यस अभ्यासमा हामीले विकिपेडिया पाठबाट केही अर्थ निकाल्ने साधारण प्रक्रिया देख्यौं, कीवर्डहरू र शब्द बादलको रूपमा। यो उदाहरण धेरै साधारण छ, तर यसले राम्रोसँग देखाउँछ ती सबै सामान्य चरणहरू जुन डाटा वैज्ञानिकले डाटासँग काम गर्दा लिन्छन्, डाटा प्राप्ति देखि लिएर दृश्यकरण सम्म।

हाम्रो पाठ्यक्रममा हामी ती सबै चरणहरूलाई विस्तारमा छलफल गर्नेछौं। 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी सही हुन प्रयास गर्छौं, तर कृपया जानकार हुनुस् कि स्वचालित अनुवादमा त्रुटिहरू वा अशुद्धताहरू हुन सक्छन्। मूल दस्तावेज़ यसको मूल भाषामा आधिकारिक स्रोत मानिनुपर्छ। महत्वपूर्ण जानकारीका लागि व्यावसायिक मानव अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न कुनै पनि गलत बुझाइ वा त्रुटिको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
